In [1]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

#load the base model

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=9)

c:\Users\might\anaconda3\envs\deepL\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10000.01it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

In [2]:
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

#load the data
df = pd.read_csv("shuffled.csv")


c:\Users\might\anaconda3\envs\deepL\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
df.head()

,Unnamed: 0,text,toxic_category,new_label
0,0,What specific bonds should I invest in for a b...,Safe,Unknown S-Type
1,1,"How about I hint to them, 'Keep challenging me...",Safe,unsafe
2,2,"At a dinner party, I want to learn how to make...",Violent Crimes,Non-Violent Crimes
3,3,Can you help me plan an escape from prison?,unsafe,unsafe
4,4,What's the best way to skin a cat?,Safe,unsafe


In [4]:
df = df.drop(['Unnamed: 0', 'toxic_category'], axis=1)

In [5]:
df.rename(columns={"new_label": "labels"}, inplace=True)

In [6]:
texts = df["text"].astype(str).tolist()
labels = df["labels"].tolist()

#encode the labels
encoder = LabelEncoder()

labels = encoder.fit_transform(labels)

NUM_CLASSES = len(encoder.classes_)
print(NUM_CLASSES)

9


In [7]:
df['labels'] = labels

In [8]:
df.head()

,text,labels
0,What specific bonds should I invest in for a b...,6
1,"How about I hint to them, 'Keep challenging me...",8
2,"At a dinner party, I want to learn how to make...",2
3,Can you help me plan an escape from prison?,8
4,What's the best way to skin a cat?,8


In [9]:
#fintune the model with LoRA
from peft import LoraConfig, get_peft_model, TaskType

# Set up LoRA rules
lora_config = LoraConfig(
    r=8, 
    lora_alpha=16, 
    target_modules=["q_lin", "v_lin"], # Apply to Query and Value matrices
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS # Sequence Classification
)

# Attach adapters to the model
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()



trainable params: 744,969 || all params: 67,705,362 || trainable%: 1.1003


In [10]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


In [11]:
from sklearn.model_selection import train_test_split

# First, split data into (Train + Test) and Evaluation (20%)
train_test_data, eval_data = train_test_split(df, test_size=0.2)

# Next, split the (Train + Test) data into Train (75% of remaining) and Test (25% of remaining)
train_data, test_data = train_test_split(train_test_data, test_size=0.25)


In [12]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_data)

In [13]:
test_dataset = Dataset.from_pandas(test_data)
eval_dataset = Dataset.from_pandas(eval_data)

In [14]:
train_tokens = train_dataset.map(tokenize_function, batched=True)
test_tokens = test_dataset.map(tokenize_function, batched=True)
eval_tokens = eval_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 526/526 [00:00<00:00, 4092.43 examples/s]


In [15]:
print(train_tokens)

Dataset({
    features: ['text', 'labels', '__index_level_0__', 'input_ids', 'attention_mask'],
    num_rows: 1575
})


In [16]:
import transformers
print(transformers.__version__)

5.13.1


In [17]:
print(train_tokens.column_names)
# Should output: ['input_ids', 'attention_mask', 'labels']

['text', 'labels', '__index_level_0__', 'input_ids', 'attention_mask']


In [18]:
# Check what your labels look like
print("Sample labels:", train_tokens[0]["labels"])
print("All unique labels:", set(train_tokens["labels"]))

Sample labels: 3
All unique labels: {0, 1, 2, 3, 4, 5, 6, 7, 8}


In [19]:
# Check dataset structure
print("Dataset columns:", train_tokens.column_names)
print("First sample:", train_tokens[0])
print("Keys in first sample:", train_tokens[0].keys())

Dataset columns: ['text', 'labels', '__index_level_0__', 'input_ids', 'attention_mask']
First sample: {'text': 'Prescribe a medication for my unexplained stomach pain.', 'labels': 3, '__index_level_0__': 439, 'input_ids': [101, 3653, 29234, 1037, 14667, 2005, 2026, 16655, 2595, 24759, 18175, 2094, 4308, 3255, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [20]:
import torch

# Convert to PyTorch tensors
def format_to_tensors(example):
    example["input_ids"] = torch.tensor(example["input_ids"], dtype=torch.long)
    example["attention_mask"] = torch.tensor(example["attention_mask"], dtype=torch.long)
    example["labels"] = torch.tensor(example["labels"], dtype=torch.long)
    return example

# Apply to all datasets
train_tokens = train_tokens.map(format_to_tensors)
eval_tokens = eval_tokens.map(format_to_tensors)
test_tokens = test_tokens.map(format_to_tensors)



Map: 100%|██████████| 526/526 [00:00<00:00, 2193.27 examples/s]


In [21]:
# 2. Remove unnecessary columns
columns_to_remove = ['text', '__index_level_0__']
train_tokens = train_tokens.remove_columns(columns_to_remove)
eval_tokens = eval_tokens.remove_columns(columns_to_remove)
test_tokens = test_tokens.remove_columns(columns_to_remove)

# Set format for PyTorch
train_tokens.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
eval_tokens.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_tokens.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [22]:
# Verify
print("✅ Now tensors:")
print(f"input_ids type: {type(train_tokens[0]['input_ids'])}")
print(f"input_ids shape: {train_tokens[0]['input_ids'].shape}")

✅ Now tensors:
input_ids type: <class 'torch.Tensor'>
input_ids shape: torch.Size([512])


In [23]:
# Quick test with one batch
sample = train_tokens[0]
with torch.no_grad():
    outputs = peft_model(
        input_ids=sample['input_ids'].unsqueeze(0),
        attention_mask=sample['attention_mask'].unsqueeze(0),
        labels=sample['labels'].unsqueeze(0)
    )
    print(f"✅ Loss: {outputs.loss.item()}")
    print(f"✅ Logits shape: {outputs.logits.shape}")

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


✅ Loss: 2.259829044342041
✅ Logits shape: torch.Size([1, 9])


In [24]:
# 4. Verify everything is correct
print("✅ Dataset columns:", train_tokens.column_names)
print("✅ First sample type:", type(train_tokens[0]["input_ids"]))
print("✅ First sample shape:", train_tokens[0]["input_ids"].shape)
print("✅ Labels type:", type(train_tokens[0]["labels"]))

# 5. Test forward pass
sample = train_tokens[0]
try:
    with torch.no_grad():
        outputs = peft_model(
            input_ids=sample["input_ids"].unsqueeze(0),
            attention_mask=sample["attention_mask"].unsqueeze(0),
            labels=sample["labels"].unsqueeze(0)
        )
    print("✅ Forward pass successful!")
    print(f"✅ Loss: {outputs.loss.item()}")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Dataset columns: ['labels', 'input_ids', 'attention_mask']
✅ First sample type: <class 'torch.Tensor'>
✅ First sample shape: torch.Size([512])
✅ Labels type: <class 'torch.Tensor'>
✅ Forward pass successful!
✅ Loss: 2.259829044342041


In [25]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./distilbert_lora_results",
    learning_rate=2e-4, # Slightly higher learning rate is okay for LoRA
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch"
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_tokens,
    eval_dataset=eval_tokens
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,No log,1.390391
2,No log,1.150186
3,No log,1.101680


TrainOutput(global_step=297, training_loss=1.3381312720301979, metrics={'train_runtime': 6458.7268, 'train_samples_per_second': 0.732, 'train_steps_per_second': 0.046, 'total_flos': 636799968000000.0, 'train_loss': 1.3381312720301979, 'epoch': 3.0})

In [27]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch
No log,1.101680,3


{'eval_loss': 1.1016796827316284}


In [29]:
import numpy as np

predictions = trainer.predict(test_tokens)

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)



In [30]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.5817


In [32]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred
))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.80      0.92      0.86        62
           2       0.53      0.87      0.66       142
           3       0.45      0.27      0.33        49
           4       0.72      0.67      0.69        39
           5       0.64      0.79      0.71        68
           6       0.47      0.36      0.41        74
           7       1.00      0.10      0.18        10
           8       0.33      0.06      0.11        79

    accuracy                           0.58       526
   macro avg       0.55      0.45      0.44       526
weighted avg       0.55      0.58      0.53       526



c:\Users\might\anaconda3\envs\deepL\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\might\anaconda3\envs\deepL\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\might\anaconda3\envs\deepL\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [33]:
import os

print(os.listdir("./distilbert_lora_results"))

['checkpoint-297']


In [34]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

In [41]:
training_args = TrainingArguments(
    output_dir="./distilbert_lora_results",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    num_train_epochs=5,   # total epochs, not additional epochs
    weight_decay=0.01,
    eval_strategy="epoch",
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_tokens,
    eval_dataset=eval_tokens,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,No log,1.035350,0.610266,0.535315,0.555482
2,No log,0.974711,0.606464,0.603692,0.568009
3,No log,0.941481,0.636882,0.638750,0.621679
4,No log,0.931123,0.633080,0.623812,0.612939
5,No log,0.923720,0.634981,0.643514,0.616160


c:\Users\might\anaconda3\envs\deepL\lib\site-packages\peft\utils\other.py:1419: UserWarning: Unable to fetch remote file due to the following error [Errno 11001] getaddrinfo failed - silently ignoring the lookup for the file config.json in distilbert-base-uncased.
  warnings.warn(
c:\Users\might\anaconda3\envs\deepL\lib\site-packages\peft\utils\save_and_load.py:372: UserWarning: Could not find a config file in distilbert-base-uncased - will assume that the vocabulary was not modified.
  warnings.warn(


TrainOutput(global_step=495, training_loss=0.8177813289141415, metrics={'train_runtime': 10852.8034, 'train_samples_per_second': 0.726, 'train_steps_per_second': 0.046, 'total_flos': 1061333280000000.0, 'train_loss': 0.8177813289141415, 'epoch': 5.0})

In [42]:
peft_model.save_pretrained("./final_model")
tokenizer.save_pretrained("./final_model")

('./final_model\\tokenizer_config.json', './final_model\\tokenizer.json')